# 1. Setup and import libraries

In [25]:
!pip install beautifulsoup4


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
# Import thư viện
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import re
from datetime import datetime, timedelta
import os
import time
import urllib.parse

# Tạo session với Header để tránh bị chặn
session = requests.Session()
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}
session.headers.update(headers)

# Setup thư mục lưu kết quả
BASE_DIR = os.path.join(os.getcwd(), "vnexpress-results")
os.makedirs(BASE_DIR, exist_ok=True)
print(f"✅ Thư mục lưu trữ: {BASE_DIR}")

✅ Thư mục lưu trữ: d:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\vnexpress-results


# 2. Helper functions

In [27]:
def parse_vnexpress_date(date_str):
    """
    VnExpress thường có chuỗi ngày dạng: 'Thứ sáu, 19/4/2024, 14:00 (GMT+7)'
    Hàm này dùng Regex để lấy đúng '19/4/2024' và chuyển thành đối tượng datetime.
    """
    try:
        # Tìm cụm ngày tháng năm dạng dd/mm/yyyy
        match = re.search(r'(\d{1,2}/\d{1,2}/\d{4})', date_str)
        if match:
            date_clean = match.group(1)
            return datetime.strptime(date_clean, "%d/%m/%Y")
    except Exception as e:
        print(f"Lỗi parse ngày: {e}")
    return None

def fetch_vnexpress_replies(article_id, parent_comment_id, session, max_replies=100):
    """
    Gọi API phụ của VnExpress để lấy toàn bộ reply của một comment mẹ.
    """
    api_reply_url = f"https://usi-saas.vnexpress.net/index/getreplay?siteid=1000000&objectid={article_id}&objecttype=1&id={parent_comment_id}&limit={max_replies}&offset=0"
    
    extracted_replies = []
    try:
        res = session.get(api_reply_url, timeout=10)
        if res.status_code == 200:
            data = res.json()
            if data.get('error') == 0 and 'data' in data and 'items' in data['data']:
                replies_raw = data['data']['items']
                
                for r in replies_raw:
                    extracted_replies.append({
                        'comment_id': r.get('comment_id'),
                        'parent_id': parent_comment_id,  # Gắn ID của comment mẹ vào
                        'is_reply': 1,                   # Đánh dấu đây là reply
                        'author': r.get('full_name'),
                        'text': r.get('content'),
                        'likes': r.get('userlike', 0),
                        'created_at': datetime.fromtimestamp(r.get('creation_time')).strftime('%Y-%m-%d %H:%M:%S') if r.get('creation_time') else None
                    })
    except Exception as e:
        print(f"      ⚠️ Lỗi khi cào reply của comment {parent_comment_id}: {e}")
        
    return extracted_replies

def fetch_vnexpress_comments(article_id, session, max_comments=100):
    """
    Lấy comment mẹ, sau đó tự động quét và cào luôn các reply bên trong.
    """
    api_url = f"https://usi-saas.vnexpress.net/index/get?offset=0&limit={max_comments}&objectid={article_id}&objecttype=1&siteid=1000000"
    all_extracted_comments = []
    
    try:
        res = session.get(api_url, timeout=10)
        if res.status_code == 200:
            data = res.json()
            if data.get('error') == 0 and 'data' in data and 'items' in data['data']:
                comments_raw = data['data']['items']
                
                for c in comments_raw:
                    comment_id = c.get('comment_id')
                    
                    # 1. Đóng gói Comment mẹ
                    all_extracted_comments.append({
                        'comment_id': comment_id,
                        'parent_id': None,  # Comment mẹ không có parent
                        'is_reply': 0,
                        'author': c.get('full_name'),
                        'text': c.get('content'),
                        'likes': c.get('userlike', 0),
                        'created_at': datetime.fromtimestamp(c.get('creation_time')).strftime('%Y-%m-%d %H:%M:%S') if c.get('creation_time') else None
                    })
                    
                    # 2. Kiểm tra xem comment mẹ này có reply không
                    # VnExpress thường có trường 'replys' (chứa 1 mảng nhỏ) hoặc tổng số reply
                    has_replies = False
                    if 'replys' in c and isinstance(c['replys'], dict) and 'items' in c['replys']:
                        if len(c['replys']['items']) > 0:
                            has_replies = True
                    
                    # 3. Nếu có reply, gọi hàm fetch_vnexpress_replies để cào vét sạch
                    if has_replies:
                        replies = fetch_vnexpress_replies(article_id, comment_id, session)
                        all_extracted_comments.extend(replies) # Gộp luôn vào mảng tổng
                        time.sleep(0.5) # Nghỉ nửa giây tránh bị server VnExpress block
                        
    except Exception as e:
        print(f"⚠️ Lỗi khi lấy comment của ID {article_id}: {e}")
    
    return all_extracted_comments


def get_vnexpress_links_by_keyword(keyword, session, max_links=100):
    """
    Quét qua nhiều trang tìm kiếm để lấy đủ số lượng link bài báo yêu cầu.
    """
    encoded_kw = urllib.parse.quote_plus(keyword)
    links_found = []
    page = 1 # Bắt đầu từ trang 1
    
    print(f"🔍 Bắt đầu quét tìm kiếm từ khóa: '{keyword}'...")
    
    # Chạy vòng lặp cho đến khi lấy đủ link hoặc hết trang
    while len(links_found) < max_links:
        # Nối thêm tham số &page= vào cuối URL
        search_url = f"https://timkiem.vnexpress.net/?q={encoded_kw}&page={page}"
        print(f"   -> Đang cào Trang {page}...")
        
        try:
            res = session.get(search_url, timeout=15)
            soup = BeautifulSoup(res.text, 'html.parser')
            title_tags = soup.find_all(class_='title-news')
            
            # Nếu trang này không có bài báo nào (tức là đã quét hết kết quả), thì dừng lại
            if not title_tags:
                print("   ⚠️ Đã quét đến trang cuối cùng. Không còn kết quả nào mới.")
                break
                
            for tag in title_tags:
                a_tag = tag.find('a')
                if a_tag and 'href' in a_tag.attrs:
                    link = a_tag['href']
                    
                    # Lọc bỏ link video/podcast
                    if 'video.vnexpress.net' not in link and 'podcast.vnexpress.net' not in link:
                        if link not in links_found:
                            links_found.append(link)
                            
                    # Dừng nếu đã đủ số lượng
                    if len(links_found) >= max_links:
                        break
            
            # Sang trang tiếp theo
            page += 1
            time.sleep(1) # Nghỉ 1 nhịp trước khi lật trang để tránh bị block
            
        except Exception as e:
            print(f"❌ Lỗi khi cào trang {page}: {e}")
            break
            
    print(f"✅ HOÀN TẤT QUÉT: Gom được tổng cộng {len(links_found)} link bài báo.")
    return links_found
        
        

# 3. Function to crawl a single post

In [28]:
def crawl_vnexpress_article(url, session, days_limit=60):
    """
    Cào nội dung và comment của 1 bài báo. 
    Lọc bỏ nếu bài báo cũ hơn giới hạn ngày (days_limit).
    """
    print(f"Đang xử lý: {url}")
    try:
        res = session.get(url, timeout=15)
        soup = BeautifulSoup(res.text, 'html.parser')
        
        # 1. Lấy ngày đăng để kiểm tra giới hạn thời gian
        date_tag = soup.find('span', class_='date')
        if not date_tag:
            print("❌ Không tìm thấy ngày đăng (có thể là video hoặc dạng bài e-magazine). Bỏ qua.")
            return None
            
        date_str = date_tag.text
        published_date = parse_vnexpress_date(date_str)
        
        # Kểm tra thời gian (Ví dụ: lọc bài trong 60 ngày gần nhất)
        if published_date:
            days_diff = (datetime.now() - published_date).days
            if days_diff > days_limit:
                print(f"⏩ Bỏ qua: Bài viết quá cũ ({days_diff} ngày trước). Limit là {days_limit} ngày.")
                return None
        
        # 2. Lấy ID bài báo (thường nằm ở meta tag 'tt_article_id')
        article_id_tag = soup.find('meta', attrs={'name': 'tt_article_id'})
        if not article_id_tag:
            print("❌ Không tìm thấy ID bài viết.")
            return None
        article_id = article_id_tag['content']
        
        # 3. Lấy Tiêu đề
        title = soup.find('h1', class_='title-detail').text.strip() if soup.find('h1', class_='title-detail') else "No Title"
        
        # 4. Lấy Nội dung (Gộp tất cả các thẻ <p> class 'Normal' lại)
        paragraphs = soup.find_all('p', class_='Normal')
        content = "\n".join([p.text.strip() for p in paragraphs])
        
        # 5. Lấy Comments
        comments = fetch_vnexpress_comments(article_id, session)
        
        # Đóng gói dữ liệu
        article_data = {
            'article_id': article_id,
            'url': url,
            'title': title,
            'published_date': published_date.strftime('%Y-%m-%d') if published_date else date_str,
            'content': content,
            'total_comments_fetched': len(comments),
            'comments': comments
        }
        
        print(f"✅ Thành công! Tiêu đề: {title[:40]}... | Lấy được {len(comments)} comments.")
        return article_data
        
    except Exception as e:
        print(f"❌ Lỗi cào link {url}: {e}")
        return None

# 4. Export to CSV files


In [29]:
def export_vnexpress_articles_to_csv(all_articles, out_dir):
    """Export article metadata to CSV"""
    os.makedirs(out_dir, exist_ok=True)
    
    df = pd.DataFrame(all_articles)
    csv_path = os.path.join(out_dir, "vnexpress_articles_metadata.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8")
    print(f"✅ Saved articles metadata CSV: {csv_path}")
    return csv_path, df


def export_vnexpress_comments_to_csv(all_comments, out_dir):
    """Export comments and replies to CSV"""
    os.makedirs(out_dir, exist_ok=True)
    
    df = pd.DataFrame(all_comments)
    # Reorder columns for better readability
    cols = ['article_id', 'comment_id', 'parent_id', 'is_reply', 'author', 'text', 'likes', 'created_at']
    df = df[[c for c in cols if c in df.columns]]
    
    csv_path = os.path.join(out_dir, "all_comments.csv")
    df.to_csv(csv_path, index=False, encoding="utf-8")
    print(f"✅ Saved comments CSV: {csv_path}")
    return csv_path, df


In [30]:
# ==========================================
# CẤU HÌNH TOOL
# ==========================================
KEYWORD = "iran" 
MAX_ARTICLES = 2  
DAYS_LIMIT = 60           
print(f"{'='*60}")
print(f"🚀 BẮT ĐẦU CÀO TỰ ĐỘNG TỪ KHÓA: {KEYWORD.upper()}")
print(f"{'='*60}\n")

# 1. TỰ ĐỘNG LẤY LINK
urls_to_crawl = get_vnexpress_links_by_keyword(KEYWORD, session, max_links=MAX_ARTICLES)

# 2. TIẾN HÀNH CÀO TỪNG BÀI
all_articles_data = []
all_comments_data = []

for idx, url in enumerate(urls_to_crawl, 1):
    print(f"\n[{idx}/{len(urls_to_crawl)}] Xử lý bài...")
    # Gọi hàm cào bài viết
    data = crawl_vnexpress_article(url, session, days_limit=DAYS_LIMIT)
    
    if data:
        # Tách metadata bài viết
        article_meta = {k: v for k, v in data.items() if k != 'comments'}
        all_articles_data.append(article_meta)
        
        # Thêm article_id vào mỗi comment
        for cmt in data['comments']:
            cmt['article_id'] = data['article_id']
            all_comments_data.append(cmt)
        
        # In thử vài comment
        if data['comments']:
            print("   💬 Trích xuất thử một vài bình luận:")
            for cmt in data['comments'][:3]:
                short_text = cmt['text'][:60].replace('\n', ' ') + "..." if len(cmt['text']) > 60 else cmt['text'].replace('\n', ' ')
                print(f"      👤 {cmt['author']}: {short_text} (👍 {cmt['likes']})")
        else:
            print("   ⚠️ Bài viết này không có bình luận nào.")
            
    time.sleep(1.5) # Rất quan trọng: Nghỉ 1.5s để web không block bạn là bot

# ==========================================
# XUẤT RA CSV
# ==========================================
print(f"\n{'='*60}")
print("📁 EXPORTING CSV FILES...")
print(f"{'='*60}")

if all_articles_data:
    # Export articles metadata
    articles_csv_path, df_articles = export_vnexpress_articles_to_csv(all_articles_data, BASE_DIR)
    print(f"📄 Saved {len(df_articles)} articles metadata")
    
    # Export all comments
    if all_comments_data:
        comments_csv_path, df_comments = export_vnexpress_comments_to_csv(all_comments_data, BASE_DIR)
        print(f"💬 Saved {len(df_comments)} total comments")
    
    print(f"\n{'='*60}")
    print("✅ HOÀN THÀNH TOÀN BỘ QUÁ TRÌNH!")
    print(f"📁 All data saved to: {BASE_DIR}")
    print(f"{'='*60}")
else:
    print("\n⚠️ Không có dữ liệu nào được cào thành công (có thể do quá hạn 60 ngày hoặc link lỗi).")


🚀 BẮT ĐẦU CÀO TỰ ĐỘNG TỪ KHÓA: IRAN

🔍 Bắt đầu quét tìm kiếm từ khóa: 'iran'...
   -> Đang cào Trang 1...
✅ HOÀN TẤT QUÉT: Gom được tổng cộng 2 link bài báo.

[1/2] Xử lý bài...
Đang xử lý: https://vnexpress.net/iran-mo-eo-bien-hormuz-5063877.html
✅ Thành công! Tiêu đề: Iran mở eo biển Hormuz... | Lấy được 12 comments.
   💬 Trích xuất thử một vài bình luận:
      👤 cameraman79dl: Tin vui! Mong rằng mọi người trên thế giới tiếp tục được cùn... (👍 81)
      👤 tvkhoi1987: Thực sự rất vui mừng! (👍 42)
      👤 thd: Hy vọng vậy. (👍 30)

[2/2] Xử lý bài...
Đang xử lý: https://vnexpress.net/my-iran-dam-phan-that-bai-5061456.html
❌ Không tìm thấy ngày đăng (có thể là video hoặc dạng bài e-magazine). Bỏ qua.

📁 EXPORTING CSV FILES...
✅ Saved articles metadata CSV: d:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\vnexpress-results\vnexpress_articles_metadata.csv
📄 Saved 1 articles metadata
✅ Saved comments CSV: d:\Study\UIT\Third-year\S2\DeepLearning\Crawdata\vnexpress-results\all_comments.csv
💬 